# Lecture 30 — Build a Tiny Neural Network From Scratch

**Source of truth:** `blog.md`

This notebook is a computational reconstruction of that exact blog. Every major idea below is kept in the blog’s terminology and context.

**Learning loop:** story → intuition → hand calculation → NumPy → PyTorch → visualization → one-variable experiment → intentional failure → blog exercises → mini-project → research.


## 📖 Blog context — 🧪 Interactive Lab

## 🧪 Interactive Lab

The matching notebook is the complete hands-on laboratory for this lesson. It contains the runnable code, experiments, visualizations, and challenges.

**[📓 Open the notebook on GitHub](https://github.com/manish7725/deeplearning/blob/main/Lecture%2030%20-%20Build%20a%20Tiny%20Neural%20Network%20From%20Scratch/notebook.ipynb)**  · **[▶ Open the notebook in Google Colab](https://colab.research.google.com/github/manish7725/deeplearning/blob/main/Lecture%2030%20-%20Build%20a%20Tiny%20Neural%20Network%20From%20Scratch/notebook.ipynb)**


## 📖 Blog context — 🧭 Where this lesson fits

## 🧭 Where this lesson fits

**Previous lesson:** Blog 19 — How Can a Machine Create Something New?.

**Today:** Blog 20 — Build a Tiny Neural Network From Scratch.

**Next lesson:** Blog 21 — How Does a Computer Calculate Gradients Automatically?.

**Student rule:** if you cannot explain why this lesson follows the previous one, stop and reread the final takeaway of the previous blog. The equations below should feel like a continuation, not a new language.


## 📖 Blog context — 1. Our toy problem

## 1. Our toy problem

Suppose the data follows:

```math
y = 2x + 1
```

Use four examples:

```python
import numpy as np

x = np.array([1., 2., 3., 4.])
y = np.array([3., 5., 7., 9.])
```

We want our model

```math
\hat{y} = wx + b
```

to discover $w \approx 2$ and $b \approx 1$.

---


## 📖 Blog context — 2. Start with terrible parameters

## 2. Start with terrible parameters

```python
w = 0.0
b = 0.0
```

The initial model predicts zero for every input.

For $x = 1$:

```math
\hat{y} = 0
```

while the correct answer is 3.

The model needs to learn.

---


## 📖 Blog context — 3. Define the loss

## 3. Define the loss

Use mean squared error (MSE):

```math
L = \frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i-y_i)^2
```

The smaller the loss, the closer our predictions are to the targets.

---


## 📖 Blog context — 4. Derive the gradients

## 4. Derive the gradients

We have

```math
\hat{y}_i = wx_i + b
```

and

```math
L = \frac{1}{n}\sum_i(\hat{y}_i-y_i)^2
```

Using the chain rule, we get the gradient with respect to the weight:

```math
\frac{\partial L}{\partial w}
= \frac{2}{n}\sum_i(\hat{y}_i-y_i)x_i
```

And the gradient with respect to the bias:

```math
\frac{\partial L}{\partial b}
= \frac{2}{n}\sum_i(\hat{y}_i-y_i)
```

These are the exact instructions needed to improve $w$ and $b$.


## 📖 Blog context — Why does the bias gradient not contain $x_i$?

### Why does the bias gradient not contain $x_i$?

Because

```math
\frac{\partial \hat{y}_i}{\partial b} = 1
```

while

```math
\frac{\partial \hat{y}_i}{\partial w} = x_i
```

That small difference is important: the weight gradient is scaled by the input, while the bias gradient is not.

---


## 📖 Blog context — 5. Write gradient descent

## 5. Write gradient descent

For learning rate $\eta$:

```math
w \leftarrow w - \eta\frac{\partial L}{\partial w}
```

```math
b \leftarrow b - \eta\frac{\partial L}{\partial b}
```

That is the entire learning algorithm for this toy model.

---


## 📖 Blog context — 6. Full NumPy implementation

## 6. Full NumPy implementation

```python
import numpy as np

x = np.array([1., 2., 3., 4.])
y = np.array([3., 5., 7., 9.])

w = 0.0
b = 0.0
learning_rate = 0.01

for step in range(2000):
    # Forward pass
    prediction = w * x + b

    # Loss
    error = prediction - y
    loss = np.mean(error ** 2)

    # Backward pass
    dw = np.mean(2 * error * x)
    db = np.mean(2 * error)

    # Update
    w -= learning_rate * dw
    b -= learning_rate * db

    if step % 200 == 0:
        print(step, loss, w, b)

print("final weight:", w)
print("final bias:", b)
```

After training, the learned parameters should be close to:

```math
w = 2, \qquad b = 1
```

The exact numerical values depend on the learning rate and number of steps.

---


## 📖 Blog context — 7. What just happened?

## 7. What just happened?

Every iteration followed the same scientific loop:

```mermaid
flowchart TD
    A[Input x] --> B[Prediction wx + b]
    B --> C[Error prediction - y]
    C --> D[Loss]
    D --> E[Gradients dw and db]
    E --> F[Update w and b]
    F --> B
```

Nothing mysterious happened.

The model started with poor parameters and repeatedly changed them according to the gradient.


## 📖 Blog context — The training loop in one picture

### The training loop in one picture

```text
             ┌───────────────┐
             │    Input x    │
             └───────┬───────┘
                     ↓
             ┌───────────────┐
             │  wx + b        │
             │  Prediction    │
             └───────┬───────┘
                     ↓
             ┌───────────────┐
             │     Loss       │
             └───────┬───────┘
                     ↓
             ┌───────────────┐
             │   Gradients    │
             │    dw, db      │
             └───────┬───────┘
                     ↓
             ┌───────────────┐
             │ Update w and b │
             └───────┬───────┘
                     │
                     └──────────→ repeat
```

---


## 📖 Blog context — 8. Now let PyTorch do the bookkeeping

## 8. Now let PyTorch do the bookkeeping

The same model can be written with PyTorch:

```python
import torch

x = torch.tensor([1., 2., 3., 4.])
y = torch.tensor([3., 5., 7., 9.])

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

optimizer = torch.optim.SGD([w, b], lr=0.01)

for step in range(2000):
    prediction = w * x + b
    loss = torch.mean((prediction - y) ** 2)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(w.item(), b.item())
```

Compare this with the NumPy version.

The mathematics is the same.

PyTorch automates the gradient calculation and parameter update machinery.

---


## 📖 Blog context — 9. From one neuron to a network

## 9. From one neuron to a network

Our model had one input and one output.

A real neural network may have:

```math
\mathbf{h} = \sigma(W_1\mathbf{x} + \mathbf{b}_1)
```

followed by

```math
\hat{y} = W_2\mathbf{h} + \mathbf{b}_2
```

Training still follows the same conceptual loop:

```math
\text{forward}
\rightarrow
\text{loss}
\rightarrow
\text{backward}
\rightarrow
\text{update}
```

The network becomes more complicated, but the core idea does not disappear.

---


## 📖 Blog context — 10. What you have actually learned

## 10. What you have actually learned

You now have the mathematical foundation needed to understand many deep-learning systems.


## 📖 Blog context — Representation

### Representation

```math
\mathbf{x}
```


## 📖 Blog context — Transformation

### Transformation

```math
W\mathbf{x} + \mathbf{b}
```


## 📖 Blog context — Nonlinearity

### Nonlinearity

```math
\sigma(W\mathbf{x} + \mathbf{b})
```


## 📖 Blog context — Prediction

### Prediction

```math
\hat{y} = f_\theta(x)
```


## 📖 Blog context — Objective

### Objective

```math
L(\hat{y}, y)
```


## 📖 Blog context — Gradient

### Gradient

```math
\nabla_\theta L
```


## 📖 Blog context — Optimization

### Optimization

```math
\theta \leftarrow \theta - \eta\nabla_\theta L
```

That is the mathematical skeleton of deep learning.

---


## 📖 Blog context — 11. Your final challenge 🧠

## 11. Your final challenge 🧠

Change the dataset to:

```math
y = 3x - 2
```

and train again.

Then try a two-feature model:

```math
\hat{y} = w_1x_1 + w_2x_2 + b
```

Derive the gradients yourself.

Then implement it.

Finally, add a hidden layer and ReLU.

At that point you will no longer be merely reading about neural networks.

You will be constructing them.

---


## 📖 Blog context — The complete mental model

## The complete mental model

```text
Real world
    ↓
Numerical representation
    ↓
Vectors / tensors
    ↓
Linear transformations
    ↓
Nonlinear transformations
    ↓
Prediction
    ↓
Loss
    ↓
Gradient
    ↓
Parameter update
    ↓
Better prediction
    ↺
```

The most important lesson is not a particular architecture.

It is this:

> **Deep learning is a way of learning useful transformations of numerical representations by optimizing a differentiable objective with data.**

Once you understand that sentence mathematically, CNNs, RNNs, Transformers and generative models stop looking like unrelated magic.

They become different ways of constructing and learning functions.

---


## 📖 Blog context — Where to go next

# Where to go next

The natural next stage is to turn this foundation into serious practice:

1. Linear algebra — vectors, matrices, eigenvalues, SVD.
2. Probability and statistics — distributions, expectation, variance and estimation.
3. Calculus — derivatives, partial derivatives and chain rule.
4. Optimization — SGD, Momentum, Adam and learning-rate schedules.
5. PyTorch — datasets, modules, autograd and training loops.
6. CNNs — image classification and computer vision.
7. Sequence models — RNN, LSTM and GRU.
8. Attention — Q, K, V and masking.
9. Transformers — encoder, decoder and modern architectures.
10. Language models — tokenization, pretraining, fine-tuning and inference.
11. Generative models — VAEs, GANs and diffusion.
12. Production deep learning — evaluation, monitoring, serving and optimization.

You have reached the end of this first-principles series.

But you have not reached the end of deep learning.

You have reached the point where the next step is to **build**.

---


## 📖 Blog context — 🧪 Final Capstone Lab — Remove Every Layer of Magic

# 🧪 Final Capstone Lab — Remove Every Layer of Magic

.

Complete the progression without skipping levels:


## 📖 Blog context — Level 1 — One parameter

### Level 1 — One parameter

Implement

$$
\hat y=wx
$$

and derive $dL/dw$.


## 📖 Blog context — Level 2 — Weight + bias

### Level 2 — Weight + bias

Implement

$$
\hat y=wx+b
$$

and derive both gradients.


## 📖 Blog context — Level 3 — Multiple features

### Level 3 — Multiple features

Implement

$$
\hat y=\mathbf w^T\mathbf x+b
$$

using vectors.


## 📖 Blog context — Level 4 — Multiple neurons

### Level 4 — Multiple neurons

Implement

$$
\mathbf z=W\mathbf x+\mathbf b
$$

using matrix multiplication.


## 📖 Blog context — Level 5 — Nonlinearity

### Level 5 — Nonlinearity

Add ReLU:

$$
\mathbf h=ReLU(W_1\mathbf x+\mathbf b_1)
$$


## 📖 Blog context — Level 6 — Backpropagation

### Level 6 — Backpropagation

Derive the gradients through the hidden layer.


## 📖 Blog context — Level 7 — PyTorch

### Level 7 — PyTorch

Rebuild the same network with `nn.Module`, autograd and an optimizer.


## 📖 Blog context — Level 8 — Explain it

### Level 8 — Explain it

Teach the entire network to another person without showing code first.

If you can do all eight levels, you have crossed an important boundary: you understand the mechanism rather than merely knowing the vocabulary.

---


## 📖 Blog context — 📚 Your Mastery Resource Stack

# 📚 Your Mastery Resource Stack

Use **3Blue1Brown** for visual mathematics and neural-network intuition.

Use **Welch Labs** for hands-on mathematical explanations, graphics and supporting code; its Neural Networks Demystified sequence and newer AI material strongly reinforce the build-and-understand approach.

Use **Frame Zero** for first-principles machine-learning intuition.

Use **MrJensenMath10** for mathematical fluency.

Use **ZacharyLLM** for the modern LLM path.

Use **Visual Kernel** for additional visual/technical understanding of model internals.


## 📖 Blog context — The final learning loop

### The final learning loop

Do not finish this series by watching more videos.

Finish it by building things.

$$
\boxed{
\text{Learn}
\rightarrow
\text{Derive}
\rightarrow
\text{Implement}
\rightarrow
\text{Experiment}
\rightarrow
\text{Break}
\rightarrow
\text{Debug}
\rightarrow
\text{Explain}
}
$$

That loop is the real curriculum.

The blogs give you the map.

The labs give you the hands.

The mathematics gives you the language.

And the experiments turn knowledge into understanding.


## ✍️ Hand calculation

**Mathematical anchor:** `forward→loss→backprop→update`

**Task:** use the smallest numerical values from the blog and calculate the result by hand. Write every intermediate step. Then verify it below.

**Why:** the notebook must reproduce the blog’s mathematics, not replace it with a generic example.

In [ ]:
import numpy as np
x=np.arange(4.).reshape(-1,1); y=2*x+1; W=np.array([[.1]]); b=np.array([0.])
for _ in range(1000):
 e=x@W+b-y; W-=.02*(2*x*e).mean(0).reshape(1,1); b-=.02*(2*e).mean(0)
print(W,b)

## 🔥 PyTorch verification

Repeat the same mathematical operation with tensors. Compare the numerical result with the NumPy/reference calculation above. For derivative chapters, also compare analytical, finite-difference, and autograd gradients.

In [ ]:
import torch
# Translate the smallest calculation above into tensors.
# Keep the numbers identical to the blog example when possible.
print(torch.tensor([1.,2.,3.]))

## 📈 Visualization

Before running the plot, predict what the mathematics says should happen. Then visualize the chapter quantity. A plot is evidence: explain its shape, direction, slope, distance, probability, or trajectory.

In [ ]:
import matplotlib.pyplot as plt
# Build a visualization from the chapter-specific values above.
# Keep it tied to the blog's mathematical question.
plt.figure(figsize=(7,4)); plt.grid(); plt.title('Mathematical prediction from the blog'); plt.show()

## 🔬 Change exactly one variable

Change ONE variable that matters to this chapter. Predict the result first, run it, and explain why it changed. Do not change several variables simultaneously.

In [ ]:
experiment_value=1.0
print('Change only experiment_value:',experiment_value)

## 💥 Intentional failure

Break the actual assumption taught in this blog. Record: **changed assumption → symptom → mathematical reason → fix**. Examples include excessive learning rate, incompatible shapes, extreme logits, data leakage, large distribution shift, wrong target, or unstable optimization—choose the one that belongs to this chapter.

In [ ]:
broken_value=None
print('Set broken_value to a deliberately bad chapter-specific value, then explain the failure.')

## 📝 Blog-synchronized exercises

These are extracted from this exact blog. They must not be replaced by unrelated exercises.

- **BLOG-20-EX-01:** ## 11. Your final challenge 🧠

## 🛠️ Mini-project

Build an extension of the blog’s central example. Include a hypothesis, implementation, visualization, one controlled variable change, one deliberate failure, and a written conclusion. If the blog specifies a project, follow that project rather than switching topics.

## 🎓 Research bridge

Formulate a falsifiable question from the blog’s main assumption: **If I change X while holding Y and Z fixed, does metric M change? Why?** Do not invent a paper citation unless the blog provides one.

## ✅ Mastery check

- Can I explain the blog’s story and intuition?
- Can I reproduce its smallest numerical example by hand?
- Can I map every important equation to code?
- Do NumPy/PyTorch implement the same idea?
- Can I predict the visualization?
- Did I change exactly one variable?
- Did I deliberately break the relevant assumption?
- Did I complete every blog exercise?
- Can I build the mini-project and formulate a research question?